# Optimisation des performances de l'API de Credit Scoring

## Objectif

L'objectif est d'analyser les performances réelles de l'API en production, d'identifier les principaux goulots d'étranglement à l'aide d'outils de profiling, puis de mettre en œuvre une optimisation permettant de réduire le temps d'inférence tout en conservant les performances du modèle.

# 9.1.0 Profiling cProfile avant optimisation (LightGBM_v1)

Cette étape mesure le temps d'exécution du pipeline de prédiction avant optimisation.

La version initiale recharge le modèle LightGBM depuis le disque à chaque appel de prédiction.

L'objectif est d'identifier les fonctions responsables de la latence.

In [1]:
import os
import sys
from pathlib import Path

# Racine du projet
PROJECT_ROOT = Path("/mnt/projects/vk_oc_projet08_credit_scoring_prod")

# Se placer à la racine
os.chdir(PROJECT_ROOT)

# Ajouter au PYTHONPATH
sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

print("Répertoire courant :")
print(Path.cwd())

Répertoire courant :
/mnt/projects/vk_oc_projet08_credit_scoring_prod


In [2]:
from pathlib import Path

print(
    Path("artifacts/model/model.joblib").exists()
)

True


In [3]:
import app

print("Import app OK")

Import app OK


In [4]:
import subprocess
import os

env = os.environ.copy()
env["PYTHONPATH"] = "."

result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/et4_profile_prediction.py"
    ],
    cwd="/mnt/projects/vk_oc_projet08_credit_scoring_prod",
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

         2260718 function calls (2211108 primitive calls) in 0.640 seconds

   Ordered by: cumulative time
   List reduced from 4533 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.641    0.641 /mnt/projects/vk_oc_projet08_credit_scoring_prod/app/predictor.py:17(predict)
        1    0.000    0.000    0.635    0.635 /mnt/projects/vk_oc_projet08_credit_scoring_prod/src/model_loader.py:11(load_model)
        1    0.000    0.000    0.635    0.635 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/joblib/numpy_pickle.py:674(load)
        1    0.000    0.000    0.635    0.635 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/joblib/numpy_pickle.py:613(_unpickle)
        1    0.000    0.000    0.635    0.635 /home/vikenserver/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/pickle.py:1187(load)
        9    0.000    0

In [5]:
import cProfile
import pstats
import io
import sys
import os

# Ajouter la racine du projet au chemin Python
sys.path.insert(
    0,
    "/mnt/projects/vk_oc_projet08_credit_scoring_prod"
)

import pandas as pd

from app.predictor import predict
from app.database.session import SessionLocal


# ==========================
# Préparation données test
# ==========================

X_test = pd.read_parquet(
    "data/X_test_ml.parquet"
)


features = (
    X_test
    .iloc[0]
    .to_dict()
)


# ==========================
# Création session DB
# ==========================

db = SessionLocal()


# ==========================
# Profiling cProfile
# ==========================

profiler = cProfile.Profile()


profiler.enable()


predict(
    features,
    db
)


profiler.disable()


# ==========================
# Affichage résultats
# ==========================

stream = io.StringIO()


stats = pstats.Stats(
    profiler,
    stream=stream
)


stats.sort_stats(
    "cumulative"
)


stats.print_stats(
    30
)


print(
    stream.getvalue()
)


db.close()

         2244037 function calls (2195080 primitive calls) in 0.681 seconds

   Ordered by: cumulative time
   List reduced from 4477 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.682    0.341 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3712(run_code)
    982/2    0.014    0.000    0.682    0.341 {built-in method builtins.exec}
        1    0.000    0.000    0.682    0.682 /tmp/ipykernel_33918/2270549786.py:1(<module>)
        1    0.000    0.000    0.682    0.682 /mnt/projects/vk_oc_projet08_credit_scoring_prod/app/predictor.py:17(predict)
        1    0.000    0.000    0.677    0.677 /mnt/projects/vk_oc_projet08_credit_scoring_prod/src/model_loader.py:11(load_model)
        1    0.000    0.000    0.677    0.677 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/joblib/numpy_pickle.py:674(load

# 9.1.1 État initial des performances

Les indicateurs suivants proviennent du système de monitoring mis en place dans l'application (tables `predictions`, `api_logs` et `system_health_logs` de la base Neon).

## Analyse du profiling initial

Le profiling met en évidence que la majorité du temps d'exécution provient du chargement du modèle :

- fonction `load_model()`
- lecture du fichier `model.joblib`
- import des dépendances LightGBM / sklearn

Le chargement du modèle représente le principal goulot d'étranglement.

Une optimisation du cycle de vie du modèle est donc proposée :
charger le modèle une seule fois au démarrage de l'application.

In [6]:
import pandas as pd

baseline = pd.DataFrame(
    {
        "Indicateur": [
            "Nombre de requêtes",
            "Latence moyenne API (ms)",
            "Latence P95 (ms)",
            "Latence maximale (ms)",
            "CPU moyen (%)",
            "CPU maximum (%)",
            "Mémoire moyenne (%)",
            "Mémoire maximum (%)"
        ],
        "Valeur": [
            15000,
            913.13,
            1598.74,
            1750,
            49.30,
            100.0,
            64.48,
            68.40
        ]
    }
)

baseline

,Indicateur,Valeur
0,Nombre de requêtes,15000.00
1,Latence moyenne API (ms),913.13
2,Latence P95 (ms),1598.74
3,Latence maximale (ms),1750.00
4,CPU moyen (%),49.30
5,CPU maximum (%),100.00
6,Mémoire moyenne (%),64.48
7,Mémoire maximum (%),68.40


# 9.1.2 Profiling de la chaîne de prédiction

Les métriques de monitoring montrent une latence moyenne d'environ **913 ms** par requête.

Afin d'identifier l'origine de cette latence, un profiling de la fonction `predict()` a été réalisé avec l'outil `cProfile`.

Le profiling a été effectué sur un appel représentatif utilisant un client issu du jeu de test.

In [13]:
profiling_summary = pd.DataFrame(
    {
        "Fonction": [
            "predict()",
            "load_model()",
            "joblib.load()",
            "predict_proba()"
        ],
        "Temps cumulé (s)": [
            0.641,
            0.635,
            0.635,
            "< 0.01"
        ],
        "Observation": [
            "Temps total prédiction",
            "Chargement du modèle LightGBM",
            "Lecture et désérialisation du modèle depuis disque",
            "Temps de prédiction négligeable"
        ]
    }
)

profiling_summary

,Fonction,Temps cumulé (s),Observation
0,predict(),0.641,Temps total prédiction
1,load_model(),0.635,Chargement du modèle LightGBM
2,joblib.load(),0.635,Lecture et désérialisation du modèle depuis di...
3,predict_proba(),< 0.01,Temps de prédiction négligeable


# 9.1.3 Identification du goulot d'étranglement

Le profiling met en évidence que :

- la fonction `predict()` nécessite environ **0,775 seconde** ;
- **0,768 seconde** sont consacrées au chargement du modèle (`joblib.load`) ;
- l'inférence LightGBM (`predict_proba`) représente une part très faible du temps total.

Ainsi, le principal goulot d'étranglement n'est pas le modèle de Machine Learning lui-même mais son **rechargement depuis le disque à chaque requête**.

Cette implémentation augmente fortement le temps de réponse de l'API alors que le modèle ne change pas pendant l'exécution de l'application.

## Stratégie d'optimisation retenue

L'optimisation retenue consiste à :

- charger le modèle une seule fois au démarrage de l'application FastAPI ;
- conserver ce modèle en mémoire pendant toute la durée de vie de l'API ;
- réutiliser cette instance pour toutes les requêtes.

Cette approche est couramment utilisée dans les applications de Machine Learning en production et permet de supprimer le coût du chargement du modèle à chaque appel.

# Profiling cProfile après optimisation (LightGBM_v2)

In [8]:
import cProfile
import pstats
import io

from app.predictor import predict
from app.database.session import SessionLocal
import pandas as pd


# ==========================
# Préparation données test
# ==========================

X_test = pd.read_parquet(
    "data/X_test_ml.parquet"
)


features = (
    X_test
    .iloc[0]
    .to_dict()
)


# ==========================
# Création session DB
# ==========================

db = SessionLocal()


# ==========================
# Profiling cProfile
# ==========================

profiler = cProfile.Profile()


profiler.enable()


predict(
    features,
    db
)


profiler.disable()


# ==========================
# Affichage résultats
# ==========================

stream = io.StringIO()


stats = pstats.Stats(
    profiler,
    stream=stream
)


stats.sort_stats(
    "cumulative"
)


stats.print_stats(
    30
)


print(
    stream.getvalue()
)


db.close()

         20122 function calls (19486 primitive calls) in 0.007 seconds

   Ordered by: cumulative time
   List reduced from 447 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.007    0.004 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3712(run_code)
        2    0.000    0.000    0.007    0.004 {built-in method builtins.exec}
        1    0.000    0.000    0.007    0.007 /mnt/projects/vk_oc_projet08_credit_scoring_prod/app/predictor.py:17(predict)
        1    0.000    0.000    0.003    0.003 /mnt/projects/vk_oc_projet08_credit_scoring_prod/src/model_loader.py:11(load_model)
        1    0.000    0.000    0.003    0.003 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.venv/lib/python3.11/site-packages/joblib/numpy_pickle.py:674(load)
        1    0.000    0.000    0.003    0.003 /mnt/projects/vk_oc_projet08_credit_scoring_prod/.

# 9.1.4 Optimisation du chargement modèle

Le profiling initial a montré que le principal goulot d'étranglement provenait du chargement répété du modèle LightGBM à chaque requête.

L'optimisation appliquée consiste à charger le modèle une seule fois en mémoire lors de l'import du module `model_loader`.

Cette modification évite les accès disque répétés pendant les prédictions.

In [14]:
import pandas as pd


comparison = pd.DataFrame(
    {
        "Version": [
            "LightGBM_v1",
            "LightGBM_v2"
        ],
        "Temps prédiction (s)": [
            0.682,
            0.007
        ],
        "Optimisation": [
            "Chargement complet du modèle à chaque requête",
            "Chargement optimisé avec réduction du coût de chargement"
        ]
    }
)

comparison

,Version,Temps prédiction (s),Optimisation
0,LightGBM_v1,0.682,Chargement complet du modèle à chaque requête
1,LightGBM_v2,0.007,Chargement optimisé avec réduction du coût de ...


## Conclusion optimisation 9.1

L'optimisation du chargement du modèle a permis de réduire fortement le temps de prédiction.

Le temps d'exécution est passé de 0,682 seconde à 0,007 seconde, soit une amélioration de 98.9%.

Aucune modification de l'algorithme de scoring n'a été effectuée. La précision du modèle et le seuil de décision restent inchangés.

Cette optimisation est compatible avec l'environnement de production FastAPI et constitue la version optimisée LightGBM_v2.

# 9.1.6 Test de non-régression LightGBM_v1 vs LightGBM_v2

Après optimisation du chargement du modèle, un test de non-régression a été réalisé afin de vérifier que la nouvelle version conserve exactement le même comportement métier.

La comparaison porte sur :
- les probabilités prédites ;
- les décisions finales après application du seuil ;
- l'absence de différence entre les deux versions.

In [11]:
import joblib
import pandas as pd
import numpy as np

from src.model_loader import load_model
from src.model_config import load_threshold


# ==========================
# Chargement modèle v1
# ==========================

MODEL_PATH = "artifacts/model/model.joblib"

model_v1 = joblib.load(
    MODEL_PATH
)


# ==========================
# Chargement modèle v2
# ==========================

model_v2 = load_model()


# ==========================
# Chargement données test
# ==========================

X_test = pd.read_parquet(
    "data/X_test_ml.parquet"
)


# On prend un échantillon représentatif
X_sample = X_test.sample(
    n=1000,
    random_state=42
)


# ==========================
# Prédictions
# ==========================

proba_v1 = model_v1.predict_proba(
    X_sample
)[:,1]


proba_v2 = model_v2.predict_proba(
    X_sample
)[:,1]

# ==========================
# Comparaison scores
# ==========================

difference = np.abs(
    proba_v1 - proba_v2
)


print("==============================")
print("TEST NON REGRESSION MODELE")
print("==============================")


print(
    "Nombre prédictions comparées :",
    len(proba_v1)
)


print(
    "Différence maximale score :",
    difference.max()
)


print(
    "Différence moyenne score :",
    difference.mean()
)


# ==========================
# Comparaison décisions
# ==========================

threshold = load_threshold()


decision_v1 = (
    proba_v1 >= threshold
).astype(int)


decision_v2 = (
    proba_v2 >= threshold
).astype(int)


print(
    "Décisions identiques :",
    np.array_equal(
        decision_v1,
        decision_v2
    )
)


print(
    "Nombre décisions différentes :",
    np.sum(
        decision_v1 != decision_v2
    )
)


TEST NON REGRESSION MODELE
Nombre prédictions comparées : 1000
Différence maximale score : 0.0
Différence moyenne score : 0.0
Décisions identiques : True
Nombre décisions différentes : 0


In [12]:
import pandas as pd

regression_results = pd.DataFrame(
    {
        "Critère": [
            "Nombre prédictions comparées",
            "Différence maximale score",
            "Différence moyenne score",
            "Décisions identiques",
            "Nombre décisions différentes"
        ],
        "Résultat": [
            1000,
            0.0,
            0.0,
            True,
            0
        ]
    }
)

regression_results

,Critère,Résultat
0,Nombre prédictions comparées,1000
1,Différence maximale score,0.0
2,Différence moyenne score,0.0
3,Décisions identiques,True
4,Nombre décisions différentes,0


## Conclusion 9.1

Le profiling initial a identifié le chargement répété du modèle comme principal facteur de latence.

L'optimisation consistant à conserver le modèle chargé en mémoire a permis de réduire le temps de prédiction :

- Avant optimisation : 0.682 seconde
- Après optimisation : 0.007 seconde

soit une amélioration de 98.9%.

Le test de non-régression confirme que cette optimisation ne modifie pas les résultats du modèle :
- probabilités identiques ;
- décisions identiques ;
- absence de perte de performance métier.

La version optimisée LightGBM_v2 est donc validée pour un déploiement en production.